In [31]:
import numpy as np
import pandas as pd

In [32]:
df = pd.read_csv("../data/final_data.csv", encoding="latin1")
df.head()

,Recency,Frequency,Monetary,CLV,TotalQuantity,UniqueProducts,ActiveDays,CustomerTenureDays,AverageUnitPrice,AverageQuantity,AverageTransactionValue,UniqueCountries
0,235,1,77183.60,0.00,74215,1,1,0,1.040000,74215.000000,77183.600000,1
1,39,5,2790.86,1519.14,2458,103,7,365,2.644011,13.505495,23.681319,1
2,158,3,1487.24,310.00,2341,22,4,282,5.764839,75.516129,57.975484,1
3,219,1,334.40,0.00,197,17,1,0,3.841176,11.588235,19.670588,1
4,171,5,1561.81,944.23,536,59,7,260,15.930706,6.305882,29.482824,1


In [33]:
df.dtypes

Recency                      int64
Frequency                    int64
Monetary                   float64
CLV                        float64
TotalQuantity                int64
UniqueProducts               int64
ActiveDays                   int64
CustomerTenureDays           int64
AverageUnitPrice           float64
AverageQuantity            float64
AverageTransactionValue    float64
UniqueCountries              int64
dtype: object

In [34]:
df.isnull().sum()

Recency                    0
Frequency                  0
Monetary                   0
CLV                        0
TotalQuantity              0
UniqueProducts             0
ActiveDays                 0
CustomerTenureDays         0
AverageUnitPrice           0
AverageQuantity            0
AverageTransactionValue    0
UniqueCountries            0
dtype: int64

In [35]:
df.duplicated().sum()

np.int64(0)

**Train Test Split**

In [36]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['CLV'])
y = df['CLV']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [37]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((2696, 11), (674, 11), (2696,), (674,))

**Preprocessing Pipeline**

In [38]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, StandardScaler

preprocessor = ColumnTransformer([
    ('log_scaling', Pipeline([
        ('log', FunctionTransformer(np.log1p)),
        ('scaler', StandardScaler())
    ]), X.columns)
])

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('log_scaling', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g.

In [39]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [40]:
print("Preprocessed X_train shape:", X_train_processed.shape)
print("Preprocessed X_test shape:", X_test_processed.shape)

Preprocessed X_train shape: (2696, 11)
Preprocessed X_test shape: (674, 11)


## Baseline Model Training

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

models = {
    "Linear Regression": LinearRegression(),
    "KNN": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(),   
    "XGBoost": XGBRegressor(), 
}

In [42]:
results = []

for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])


    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append([name, mae, rmse, r2])

results_df = pd.DataFrame(results, columns=["Model", "MAE", "RMSE", "R2"])
results_df = results_df.sort_values(by="MAE")

print("\nModel Comparison Before Tuning:")
print(results_df)


Model Comparison Before Tuning:
               Model          MAE         RMSE        R2
3      Random Forest   693.461898  6288.935861  0.392018
4            XGBoost   704.573318  7092.565751  0.226709
1                KNN   715.241033  7256.004185  0.190659
2      Decision Tree   996.223752  8139.623864 -0.018462
0  Linear Regression  1388.817513  6722.621769  0.305274


## Hyperparameter Optimization
**Bayesian-search-based optimization**

In [43]:
import optuna
from sklearn.model_selection import cross_val_score

In [44]:

def objective(trial):

    model_name = trial.suggest_categorical('model', ['LinearRegression', 'KNN', 'DecisionTree', 'RandomForest', 'XGBoost'])

    # linear regression
    if model_name == 'LinearRegression':
        model = LinearRegression()

    # knn
    elif model_name == 'KNN':
        model = KNeighborsRegressor(
            n_neighbors=trial.suggest_int('knn_n_neighbors', 3, 30),
            weights=trial.suggest_categorical('knn_weights', ['uniform', 'distance']),
            p=trial.suggest_categorical('knn_p', [1, 2]),
            leaf_size=trial.suggest_int('knn_leaf_size', 20, 50))

    # decision tree
    elif model_name == 'DecisionTree':
        model = DecisionTreeRegressor(
            max_depth=trial.suggest_categorical('dt_max_depth', [None, 5, 10, 20, 30]),
            min_samples_split=trial.suggest_int('dt_min_samples_split', 2, 20),
            min_samples_leaf=trial.suggest_int('dt_min_samples_leaf', 1, 10),
            max_features=trial.suggest_categorical('dt_max_features', [None, 'sqrt', 'log2']),
            random_state=42)

    # random forest
    elif model_name == 'RandomForest':
        model = RandomForestRegressor(
            n_estimators=trial.suggest_categorical('rf_n_estimators', [100, 200, 300, 500, 1000]),
            max_depth=trial.suggest_categorical('rf_max_depth', [None, 5, 10, 20, 30]),
            min_samples_split=trial.suggest_int('rf_min_samples_split', 2, 20),
            min_samples_leaf=trial.suggest_int('rf_min_samples_leaf', 1, 10),
            max_features=trial.suggest_categorical('rf_max_features', [1.0, 'sqrt', 'log2']),
            random_state=42,
            n_jobs=1)

    # xgboost
    elif model_name == 'XGBoost':
        model = XGBRegressor(
            n_estimators=trial.suggest_categorical('xgb_n_estimators', [100, 200, 300, 500, 1000]),
            max_depth=trial.suggest_int('xgb_max_depth', 3, 10),
            learning_rate=trial.suggest_float('xgb_learning_rate', 0.01, 0.3, log=True),
            subsample=trial.suggest_float('xgb_subsample', 0.8, 1.0),
            colsample_bytree=trial.suggest_float('xgb_colsample_bytree', 0.8, 1.0),
            min_child_weight=trial.suggest_int('xgb_min_child_weight', 1, 10),
            reg_lambda=trial.suggest_float('xgb_reg_lambda', 1e-3, 10.0, log=True),
            objective='reg:squarederror',
            random_state=42,
            n_jobs=1)

    pipeline = Pipeline([('preprocessor', preprocessor), ('regressor', model)])

    scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='neg_mean_absolute_error', n_jobs=2)

    return -scores.mean()


In [45]:
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))

[I 2026-10-17 21:26:04,195] A new study created in memory with name: no-name-85f6bedb-8250-41eb-863b-756ae5aa49a1


In [46]:
study.optimize(objective, n_trials=100)

[I 2026-10-17 21:26:08,281] Trial 0 finished with value: 428.73105007822915 and parameters: {'model': 'KNN', 'knn_n_neighbors': 7, 'knn_weights': 'distance', 'knn_p': 2, 'knn_leaf_size': 20}. Best is trial 0 with value: 428.73105007822915.
[I 2026-10-17 21:26:08,428] Trial 1 finished with value: 1088.9619113370197 and parameters: {'model': 'LinearRegression'}. Best is trial 0 with value: 428.73105007822915.
[I 2026-10-17 21:26:11,306] Trial 2 finished with value: 431.34721733596996 and parameters: {'model': 'XGBoost', 'xgb_n_estimators': 1000, 'xgb_max_depth': 4, 'xgb_learning_rate': 0.05748924681991978, 'xgb_subsample': 0.9184829137724085, 'xgb_colsample_bytree': 0.8092900825439996, 'xgb_min_child_weight': 7, 'xgb_reg_lambda': 0.004809461967501573}. Best is trial 0 with value: 428.73105007822915.
[I 2026-10-17 21:26:11,427] Trial 3 finished with value: 561.9770492708268 and parameters: {'model': 'DecisionTree', 'dt_max_depth': 5, 'dt_min_samples_split': 2, 'dt_min_samples_leaf': 10, '

In [47]:
print("Best MAE:", study.best_value)
print("Best Parameters:", study.best_params)

Best MAE: 385.88253323484486
Best Parameters: {'model': 'RandomForest', 'rf_n_estimators': 500, 'rf_max_depth': None, 'rf_min_samples_split': 5, 'rf_min_samples_leaf': 2, 'rf_max_features': 1.0}


**After hyperparameter optimization, Random Forest was selected for model training, achieving the lowest cross-validation MAE of 385.88**

In [48]:
from sklearn.compose import TransformedTargetRegressor

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=500, max_depth=None, min_samples_split=5,
        min_samples_leaf=2, max_features=1.0, random_state=42, n_jobs=-1
    ))
])

# apply log transformation to the target automatically
best_tuned_model = TransformedTargetRegressor(regressor=rf_pipeline, func=np.log1p, inverse_func=np.expm1)

best_tuned_model.fit(X_train, y_train)


,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.",Pipeline(step...m_state=42))])
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",<ufunc 'log1p'>
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",<ufunc 'expm1'>
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](11,)","['Recency','Frequency','Monetary',...,'AverageQuantity', 'AverageTransactionValue','UniqueCountries']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying regressor exposes such an attribute when fit... versionadded:: 0.24,int,11
regressor_ regressor_: objectFitted regressor.,Pipeline,Pipeline(step...m_state=42))])
transformer_ transformer_: objectTransformer used in :meth:`fit` and :meth:`predict`.,FunctionTransformer,FunctionTrans...validate=True)
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"


In [49]:
y_pred = best_tuned_model.predict(X_test)

print("Final Model: Log-Transformed Random Forest")
print("Test MAE:", mean_absolute_error(y_test, y_pred))
print("Test RMSE:", root_mean_squared_error(y_test, y_pred))
print("Test R²:", r2_score(y_test, y_pred))

Final Model: Log-Transformed Random Forest
Test MAE: 638.3449742804802
Test RMSE: 6997.010884555192
Test R²: 0.24740477507359315


In [53]:
# Checking Overfitting
y_train_pred = best_tuned_model.predict(X_train)
y_test_pred = best_tuned_model.predict(X_test)

print("Training MAE:", mean_absolute_error(y_train, y_train_pred))
print("Test MAE:", mean_absolute_error(y_test, y_test_pred))

print("Training R²:", r2_score(y_train, y_train_pred))
print("Test R²:", r2_score(y_test, y_test_pred))

Training MAE: 217.39017327601437
Test MAE: 638.3449742804801
Training R²: 0.6926447622429734
Test R²: 0.24740477507359326


In [51]:
# saving best tuned model
import joblib
joblib.dump(best_tuned_model, "../models/customer_lifetime_value_model.pkl")
joblib.dump(X_train.columns.tolist(), "../models/customer_lifetime_value_columns.pkl")

['../models/customer_lifetime_value_columns.pkl']

**I selected Random Forest as my final model for predicting customer lifetime value over a 90-day period. During the process, i tuned all 5 models using Optuna and applied log transformation to the target variable to improve its performance. my final model achieved a test MAE of 638.34, which means its predictions were off by around 638 CLV units on average. And my model is currently overfitting but i saved the model and plan to use it in a streamlit application, where users can enter customer details and get a 90-day CLV prediction. in the future, i would also try to improve the model by creating better features and reducing large prediction errors.**